# 01 — Ingestão e embeddings locais

Este notebook transforma o corpus estático do RAGnaldo em um índice vetorial persistente. O objetivo é executar a etapa pesada uma vez, fora do startup da aplicação.

## Fluxo

1. descobrir PDFs e documentos autorais;
2. extrair texto preservando fonte e página;
3. produzir chunks com sobreposição;
4. gerar embeddings locais pelo wrapper do LangChain;
5. salvar FAISS, docstore e manifesto com hashes.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ragnaldo.config import AUTHORIAL_SOURCES_DIR, RAW_DATA_DIR, SETTINGS, VECTOR_STORE_DIR
from ragnaldo.ingestion import (
    build_index,
    discover_sources,
    load_sources,
    split_documents,
)

SETTINGS

## 1. Inventário das fontes

Arquivos oficiais baixados ficam em `data/raw` e não são versionados. Os documentos autorais ficam em `docs/sources`. O HTML bruto é mantido como evidência da pesquisa, mas a primeira versão do índice usa PDF, Markdown e TXT.

In [ ]:
source_paths = discover_sources(RAW_DATA_DIR, AUTHORIAL_SOURCES_DIR)

for path in source_paths:
    print(f'{path.suffix.lower():>5}  {path.relative_to(PROJECT_ROOT)}')

print(f'\n{len(source_paths)} fontes prontas para ingestão')

### Atenção ao manual histórico

O arquivo `manual_one_historico.pdf` está disponível para pesquisa histórica, mas não deve fundamentar regras atuais. Para o primeiro índice, vamos removê-lo explicitamente do conjunto.

In [ ]:
active_sources = [
    path for path in source_paths
    if path.name != 'manual_one_historico.pdf'
]
active_sources

## 2. Carregamento

`PyPDFLoader` cria um documento por página. Os arquivos Markdown são carregados como documentos textuais. Cada documento recebe nome, caminho e SHA-256 da fonte.

In [ ]:
documents = load_sources(active_sources)
print(f'{len(documents)} documentos/páginas carregados')
documents[0].metadata

## 3. Chunking

A configuração inicial usa 1.000 caracteres com 150 de overlap. Esses valores são hipótese, não verdade universal: serão avaliados no notebook 04. Cada chunk recebe um identificador determinístico.

In [ ]:
chunks = split_documents(documents)
print(f'{len(chunks)} chunks produzidos')
print(chunks[0].metadata)
print(chunks[0].page_content[:500])

## 4. Embeddings e índice

O wrapper `HuggingFaceEmbeddings` do LangChain carrega o modelo local em CPU. FAISS usa produto interno sobre vetores normalizados, equivalente à similaridade de cosseno para comparação de ranking.

In [ ]:
manifest = build_index(chunks, destination=VECTOR_STORE_DIR)
manifest

## Resultado esperado

A pasta `artifacts/vector_store` passa a conter `index.faiss`, `index.pkl` e `artifact_manifest.json`. Os arquivos permanecem fora do Git até definirmos a estratégia de build/deploy. O manifesto permite detectar alteração antes da desserialização do docstore.